# Phase 4 — Securities Analysis (Technical Indicators)

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** real RSI/MA numbers for a stock the client actually holds.

**Data fact:** NVDA is held by **CLT-002 / CLT-005** only. CLT-003 holds no individual stocks — asking it for NVDA analysis should say 'you don't hold NVDA'.


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env', override=True)  # override=True: beat VS Code's stale cached env vars

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

## 1. Indicators compute real values (Strategy + Factory)

In [ ]:
import pandas as pd, numpy as np
from app.indicators import IndicatorFactory
close = pd.Series(np.linspace(100, 120, 60) + np.random.RandomState(0).randn(60))
rsi = IndicatorFactory().get('rsi').compute(close)
print('RSI tail:', round(float(rsi.iloc[-1]), 2))

## 2. Technical analysis on a held stock (CLT-005 / NVDA)

In [ ]:
from app.tools.indicator_tools import technical_analysis
print(technical_analysis('NVDA', ['rsi', 'sma_20', 'sma_50']))

## 3. The agent explains the numbers (does NOT invent them)

In [ ]:
from app.agents.securities_analysis import SecuritiesAnalysisAgent
sa = SecuritiesAnalysisAgent()
print(sa.answer(client_id='CLT-005',
        query='Perform a technical analysis of my NVIDIA position including moving averages and RSI'))

## 4. Cash / not-held positions are rejected cleanly

In [ ]:
print(technical_analysis('CASH', ['rsi']))  # expect 'N/A - cash position'
print(sa.answer(client_id='CLT-003', query='technical analysis of my NVDA position'))  # not held

## ✅ Acceptance check

In [ ]:
res = technical_analysis('NVDA', ['rsi'])
assert 'rsi' in str(res).lower()
print('Phase 4 acceptance: PASS (also confirm the RSI unit test passes: make test)')